# Constant-reference Muskingum–Cunge: parent calibration and validation

Alpha is fixed at **2**. Manning n is the only fitted parameter. This notebook reviews the separately prepared parent runs and automatically screened independent events. It does not rerun or promote child networks.

The fixed-X control uses the same physical grid and corrected two-time inlet. It is a controlled comparison of X formulations, not an exact reproduction of the old floored 200 m model. Read the experiment report for limitations.

**Assessment closed, 6 October 2026:** case 11 remained unresolved after slope, BASED-depth and two-zone roughness sensitivities. Fixed X and the existing BASED relationship are retained. Read the [extensive case-11 report](../outputs/cunge_case11_two_zone/case_11_final_report.html) for mechanisms, numerical checks, limitations and suggested reporting text. The two-zone test is separate from the main one-n-per-network calibration.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'RAPID/src').is_dir())
OUT = ROOT / 'RAPID/outputs/cunge_calibration_alpha2'
summary = json.loads((OUT / 'summary.json').read_text())
results = pd.read_csv(OUT / 'comparison.csv', dtype={'example': str})
display(pd.DataFrame(summary['calibration_summary']))

## Calibration candidates

Same reviewed event windows and first-sampled-maximum lag objective as the original notebook. Alpha, discharge, depth, width and slope are not fitted. Counts and clock remain fixed throughout each roughness search. A sampling-tolerance fit does not establish a unique or physically verified Manning n.

In [ ]:
display(results[['example', 'method', 'n_manning', 'abs_error_h', 'within_sampling',
                 'common_band_within_sampling', 'cap_fraction', 'normalized_shape_rmse']]
        .sort_values(['example', 'method']).round(5))
display(Image(filename=str(OUT / 'calibration_comparison.png')))

## Interactive calibration hydrographs

These are the **16 Excel-selected calibration events**. Each curve is independently min–max normalized on the y-axis; timestamps are unchanged, with a common time origin within each panel. No peak alignment or time stretching is applied.

**Grey dashed:** upstream gauge. **Black dashed:** downstream gauge. **Cyan:** fixed-X model. **Red:** Cunge model. Hover shows UTC and original discharge in m³/s. Click a legend entry to toggle that series across all panels. The figure has a fixed width to keep labels legible; narrower notebook views can scroll horizontally.

The model curves use their separately calibrated Manning n values. Normalization hides absolute-discharge differences; it does not establish good amplitude agreement. Gauge lines join available samples and do not imply complete coverage across gaps.

In [ ]:
import sys
from IPython.display import HTML
investigations = str(ROOT / 'RAPID/investigations')
if investigations not in sys.path:
    sys.path.insert(0, investigations)
from cunge_hydrograph_figure import build_figure, save_figure
hydrographs = build_figure(results)
hydrographs_html = save_figure(hydrographs, OUT)
display(HTML(hydrographs.to_html(
    full_html=False, include_plotlyjs=True,
    config={'displaylogo': False, 'responsive': False,
            'toImageButtonOptions': {'format': 'svg', 'filename': 'calibration_hydrographs'}}
)))

## Additional exploratory event screening

Events were selected using gauge data alone. All model parameters, including reference discharge, are inherited without refitting. Automated screening is not a substitute for checking event correspondence, tributary effects, rating changes and antecedent-flow sensitivity.

This additional population contains 28 automatically selected events, with 21 usable paired comparisons. It is separate from the 16 Excel-selected calibration events above.

In [ ]:
selection = pd.read_csv(OUT / 'independent_events/selection.csv', dtype={'example': str})
independent = pd.read_csv(OUT / 'independent_comparison.csv', dtype={'example': str})
display(selection[['example', 'event', 'start', 'end', 'status']])
display(independent[['example', 'event', 'abs_error_h_fixed', 'abs_error_h_cunge',
                     'normalized_shape_rmse_fixed', 'normalized_shape_rmse_cunge']].round(4))
display(Image(filename=str(OUT / 'independent_events.png')))

## Numerical checks and new prep files

All fitted parent Cunge prep bundles were routed through the sparse RAPID engine and compared with the independent native recurrence used for calibration. These checks are separate from agreement with gauges.

In [ ]:
display(pd.read_csv(OUT / 'prepared_verification.csv', dtype={'example': str}))
display(pd.read_csv(OUT / 'wave_convergence.csv', dtype={'example': str}))
display(pd.read_csv(OUT / 'fitted_pulse_checks.csv', dtype={'example': str}))
display(results.loc[results.method.eq('cunge'), ['example', 'output_dir']])

## Reproduce the assessment

Run the following commands from the repository root in the project scientific Python environment. Existing input/implementation hashes protect calibration cache reuse.

```sh
python RAPID/investigations/cunge_calibration.py --workers 2
python RAPID/investigations/cunge_validation.py select
python RAPID/investigations/cunge_validation.py independent --workers 2
python RAPID/investigations/cunge_validation.py prepared --workers 2
python RAPID/investigations/cunge_report.py
```

Each candidate directory contains `cunge_prep/`, `fixed_x_prep/`, calibrated waves, observed gauge data, trial scores and a source manifest. `cunge_run/` contains the complete sparse-engine result. The original production defaults and historical calibration remain available.